# Fading-Shepard waveforms — holding $Z$ out of plane, indefinitely

> *"…and keep it there."*

Notebook 04 stopped at a wall. A sustained axial offset costs **every** channel a permanent
chirp $\dot f_Z = Z/(2\,\text{lens\_scale})$ — 48.5 MHz/ms at 10 µm — so Eq. 1 buys about
**206 µs** of it and the schedule is written by the RF band rather than by the atoms.

This notebook removes the wall. Eqs. S24–S28 replace each channel's single tone by a *ladder*
of tones spaced $\Delta f$, all co-chirping together, each switched on only while it sits
inside a fixed window: as the ladder slides, a rung fades out at one edge and its neighbour
fades in at the other. The drive is periodic in $f_Z$ although $f_Z$ is periodic in nothing —
Shepard's endlessly rising tone, in RF.

| § | The claim |
|---|-----------|
| 1 | plain Eq. S19 refuses a 1 ms hold at 10 µm — by 41 MHz |
| 2 | the ladders, the $\cos^p$ windows (S26/S27) and the interlaced $\xi$ offsets |
| 3 | 10 µm held for a millisecond: tracking, and total power flat to 0.43 % |
| 4 | shadow tweezers at $\pm(\lambda F/v)\Delta f$, and the $(M_x{+}2)\times M_y$ array grid |
| 5 | interlaced vs simultaneous fading: the static Mach–Zehnder of Fig. S6 |
| 6 | the user story at a human pace |
| 7 | two design caveats, quantified |

Physics reference: arXiv:2510.11451 (equations `S#` refer to its Supplement); notebooks 03–04
derive the four-channel physics this page uses.

In [ ]:
import time
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm

from aodl import (
    ArraySpec,
    ChannelFade,
    Hold,
    Lift,
    ShepardConfig,
    TermArray,
    TrajectorySpec,
    Translate,
    auto_grid,
    build_terms,
    default_1030,
    f_z_ramp,
    fade_window,
    max_z_integral,
    measure,
    render_movie,
    simulate,
    synthesize,
)
from aodl.field.focal import Z_LAB_SIGN, FrameGrid, spot_params
from aodl.units import MHz, ms, um, us

P = default_1030()                       # paper hardware at lambda = 1030 nm (docs/PLAN.md 1.5)
optics = P.optics
tau = P.channels["Ax"].transit_time
OUT = Path("outputs")                    # examples/outputs/ - gitignored
COLORS = {"Ax": "#3a7bd5", "Bx": "#8ac926", "Ay": "#f4a261", "By": "#c1121f"}


def with_order(params, order):
    "The same hardware with a different weak-drive expansion order (params.py)."
    return replace(params, channels={name: replace(a, mixing_order=order)
                                     for name, a in params.channels.items()})


def select(terms, keep):
    "The sub-array of `terms` picked by a boolean mask (fill edges are frame-wide)."
    return TermArray(c=terms.c[keep], theta1=terms.theta1[:, keep], theta2=terms.theta2[:, keep],
                     alpha=terms.alpha[:, :, keep], df_opt=terms.df_opt[keep], edge=terms.edge)


def at_column(terms, x_target, tol):
    "Terms deflected to within `tol` of (x_target, 0) - the tweezer alone, without its shadows."
    xc, yc, _, _ = spot_params(terms, optics, 0.0)
    return select(terms, (np.abs(xc - x_target) < tol) & (np.abs(yc) < tol))


def fade_times(wfs, channel, level):
    "Frame times whose drive time sits where |g| = level on `channel` (one per hand-over)."
    found = [tone.env.crossing_times(level) for tone in wfs.channels[channel].tones]
    times = np.sort(np.concatenate(found)) + 0.5 * tau
    times = times[(times >= tau) & (times <= wfs.t_span[1])]
    return times[np.concatenate([[True], np.diff(times) > 1e-9])]


P1 = with_order(P, 1)                    # one tone, one beam: the Eq. S24-S28 algebra
DFX, DFY = 8.0 * MHz, 6.5 * MHz          # ladder spacings - wide, i.e. slow fades (section 7a)

# ---- the hold Eq. 1 cannot buy: 10 um out of plane, for a millisecond ----------------------
hold = TrajectorySpec(
    array=ArraySpec(1, 1),
    moves=(Lift(10 * um, 60 * us), Hold(1 * ms), Lift(-10 * um, 60 * us)),
)
wfs = synthesize(hold, P1, shepard=ShepardConfig(DFX, DFY))
# -------------------------------------------------------------------------------------------

print("rungs: " + ", ".join(f"{n}:{cw.n_tones}" for n, cw in wfs.channels.items())
      + f"   ({wfs.n_tones} tones for {wfs.t_span[1] / us:.0f} us of drive)")
print(f"description: {wfs.description}")

## 1. The problem: Eq. 1 pays for 206 µs

Table I buys the axial degree of freedom with a **co-chirp on all four channels**, so a
constant $Z$ is a constant $\dot f_Z$ on every one of them and the frequency simply walks:

$$f_Z(t) = \frac{1}{2\,\text{lens\_scale}}\int_0^t Z\,dt' \quad\Longrightarrow\quad
\Big|\int Z\,dt\Big| \;\le\; 2\,\text{lens\_scale}\,\big(f_{\max}-f_{\rm centre}\big)
= 2.06\times 10^{-9}\ \text{m·s}$$

with nothing else in the band (`docs/PLAN.md` §1.5 — the budget is one-sided because Eq. S19
starts the drive at the carrier). At $Z = 10$ µm that is 206 µs of "up time", full stop: not a
modelling limit but the deflector's bandwidth. A millisecond of it asks for five ceilings, and
the synthesizer says so with the arithmetic.

In [ ]:
_, _, z_hold = hold.compile()
f_z = f_z_ramp(z_hold, P)
ceiling = max_z_integral(P)
requested = 2 * P.lens_scale * abs(float(f_z(hold.duration)))

try:
    synthesize(hold, P1)                              # plain Eq. S19: no ladders, no fading
except ValueError as exc:
    print(exc)
print(f"\nEq. 1 ceiling    |int Z dt| <= {ceiling:.3e} m.s  (Z = 10 um for "
      f"{ceiling / (10 * um) / us:.0f} us)")
print(f"this trajectory  |int Z dt|  = {requested:.3e} m.s  ({requested / ceiling:.2f} ceilings, "
      f"f_Z walks {float(f_z(hold.duration)) / MHz:.1f} MHz)")

## 2. The scheme: a ladder, a window, and an offset

**The ladder (Eqs. S24/S25).** Every channel carries all the rungs at once,

$$f_\mu^{(n)}(t) = f_{\text{lat},\mu}(t) + f_Z(t) + (n + \xi_\mu)\,\Delta f,\qquad n\in\mathbb{Z},$$

and each rung is judged by its own **fade coordinate** $g_n = f_Z + (n+\xi_\mu)\Delta f$ — the
frequency *minus* the lateral term, so a sideways move does not disturb the fade schedule.

**The window (Eqs. S26/S27).** With duty $\eta$ (½ here), exponent $p$ and width $M$,

$$A^{(n)} = \begin{cases}
1 & |g| \le (M-\eta)\Delta f/2\\[2pt]
\cos^{p}\theta,\quad \theta = \dfrac{\pi}{2\eta}\Big(\dfrac{|g|}{\Delta f}-\dfrac{M}{2}\Big)+\dfrac{\pi}{4} & \text{in between}\\[4pt]
0 & |g| \ge (M+\eta)\Delta f/2 .\end{cases}$$

$\theta$ runs $0 \to \pi/2$ across the shoulder, so a dying rung carries $\cos^p\theta$ exactly
while its neighbour carries $\sin^p\theta$. A tweezer is a *product* of one $A$ line and one
$B$ line (Eq. S7) and its position depends only on the index **difference**, so the two
co-located combinations carry $\cos^{2(p_A+p_B)}\theta$ and $\sin^{2(p_A+p_B)}\theta$ of the
light: constant through the hand-over exactly when $p_A + p_B = 1$. Table II splits that unit
two ways — $(½, ½)$ for a single tweezer, $(1, 0)$ for an array, where the $B$ ladder **is** the
array ladder and must not be shaped at all.

**The offset.** $\xi = 0$ on the $x$ pair and $\xi = ½$ on the $y$ pair: with $\eta = ½$ and one
common spacing the $x$ fade zones tile the $y$ plateaus exactly, so only ever one axis hands
over at a time (§5 is what that buys). $B$-ladder phases follow Eq. S28,
$\varphi^{(n)} = 2\pi n(n-1)/2M$.

Left: what the four channels actually do during the hold — opacity is the envelope, so the
hand-overs are visible as the cascade of the paper's Fig. 3b. Every rung's frequency law is
unbounded (the dotted line is the *single* Eq. S19 tone, on its way out of the band); what
stays bounded is the set of rungs that are switched on.

In [ ]:
t = np.linspace(0.0, wfs.t_span[1], 700)
t_c = t - 0.5 * tau                                   # what the beam centre sees (retarded)
plain = synthesize(hold, P1, check_band=False)        # infeasible: for plotting only

fig, (ax_s, ax_w) = plt.subplots(1, 2, figsize=(12.4, 4.3), gridspec_kw={"width_ratios": [2.4, 1]})
ax_s.plot(t / us, (P.channels["Ax"].f_center
                   + np.asarray(plain.channels["Ax"].tones[0].freq(t_c))) / MHz,
          color="#8d99ae", lw=1.3, ls=":", label="plain Eq. S19 (one tone, leaves the band)")
for name, color in COLORS.items():
    aod = P.channels[name]
    # The A ladders are drawn wide and the B ladders thin on top: with no lateral motion the
    # two members of a pair carry the *same* law (Eq. S19 splits only the lateral term), so
    # Ax/Bx and Ay/By coincide exactly here.
    width = 4.5 if name.startswith("A") else 1.6
    for tone in wfs.channels[name].tones:
        f = (aod.f_center + np.asarray(tone.freq(t_c))) / MHz
        a = np.clip(np.asarray(tone.env.A(t_c)), 0.0, 1.0)
        points = np.column_stack([t / us, f])
        ax_s.add_collection(LineCollection(
            list(np.stack([points[:-1], points[1:]], axis=1)), colors=color, linewidths=width,
            alpha=0.06 + 0.94 * 0.5 * (a[:-1] + a[1:])))
    ax_s.plot([], [], color=color, lw=width, label=f"{name} ({wfs.channels[name].n_tones} rungs)")
for edge in P.channels["Ax"].band:
    ax_s.axhline(edge / MHz, color="k", lw=0.9, ls="--")
ax_s.set(xlim=(0, t[-1] / us), ylim=(88, 120), xlabel="t [µs]",
         ylabel="drive at the beam centre [MHz]",
         title="every rung rises; the live window does not (opacity = envelope)")
ax_s.legend(fontsize=8, ncols=3, loc="upper left")

g = np.linspace(-1.6, 1.6, 801)
for p, label, color in ((0.5, r"$p = 1/2$  (single tweezer)", "#3a7bd5"),
                        (1.0, r"$p = 1$  (array, $A$)", "#f4a261"),
                        (0.0, r"$p = 0$  (array, $B$ = the array ladder)", "#c1121f")):
    ax_w.plot(g, fade_window(g * DFX, DFX, p=p), color=color, lw=1.7, label=label)
# every co-located combination of the ladder: sum (A_A A_B)^2 = sum cos^{2(p_A+p_B)} = 1
ladder = sum(np.asarray(fade_window((g - n) * DFX, DFX, p=0.5)) ** 4 for n in (-2, -1, 0, 1, 2))
ax_w.plot(g, ladder, color="#8ac926", lw=1.4, ls="--",
          label=r"$\sum_n (A_AA_B)^2 = 1$  ($p_A + p_B = 1$)")
for edge in (-0.75, -0.25, 0.25, 0.75):
    ax_w.axvline(edge, color="k", lw=0.5, alpha=0.3)
ax_w.set(xlabel=r"fade coordinate $g / \Delta f$", ylabel="amplitude", ylim=(-0.05, 1.15),
         title=r"Eqs. S26/S27 windows, $\eta = 1/2$")
ax_w.legend(fontsize=7, loc="lower center")
plt.tight_layout()
plt.show()

## 3. It works: a millisecond at 10 µm

The test of a hand-over scheme is that nothing happens. Below, the tweezer is probed straight
through the hold — including at every fade centre and every plateau centre — and asked the M3
questions: does it stay put, does it sit at the requested $\bar Z$, is it astigmatism-free, and
**does its total power ripple**? The power is summed over the tweezer's own frequency groups
(the rungs handing over are not degenerate with each other, so they add in intensity), and the
shadows of §4 are excluded by selecting terms on their Table I deflection.

The residual 0.4 % is not numerical: it is the aperture-expansion correction of §7a, which
grows as the square of how fast the fade is compared with the beam transit.

In [ ]:
tol = 0.25 * P.deflection_scale * DFY
probes = np.unique(np.concatenate([
    np.linspace(tau, hold.duration, 200),
    *[fade_times(wfs, ch, level) for ch in ("Ax", "Ay")
      for level in (0.0, wfs.channels[ch].tones[0].env.g_centre)],
]))
probes = probes[probes <= hold.duration]

rows = []
for probe in probes:
    trap = measure(at_column(build_terms(wfs, float(probe)), 0.0, tol), optics)
    total = sum(m.power for m in trap)
    rows.append((total, max(abs(m.x) for m in trap), max(abs(m.y) for m in trap),
                 sum(m.power * m.z_lab for m in trap) / total,
                 max(abs(m.delta_f) for m in trap)))
power, xerr, yerr, zbar, astig = (np.array(column) for column in zip(*rows))
requested_z = np.asarray(z_hold(probes - 0.5 * tau))
flatness = (power.max() - power.min()) / power.mean()
cycles = float(f_z(hold.duration)) / DFX

print(f"hand-overs in the run   {cycles:.1f} on x, {float(f_z(hold.duration)) / DFY:.1f} on y")
print(f"total tweezer power     flat to {100 * flatness:.3f} %   "
      f"(min {power.min() / power.mean():.4f}, max {power.max() / power.mean():.4f} of the mean)")
print(f"lateral error           {max(xerr.max(), yerr.max()) / optics.waist0:.2e} waists")
print(f"axial error             {np.max(np.abs(zbar - requested_z)) / optics.rayleigh:.2e} z_R")
print(f"astigmatism             {astig.max() / optics.rayleigh:.2e} z_R")
assert flatness < 0.01
assert max(xerr.max(), yerr.max()) < 0.01 * optics.waist0
assert np.max(np.abs(zbar - requested_z)) < 0.02 * optics.rayleigh
assert astig.max() < 0.02 * optics.rayleigh

In [ ]:
scan = np.linspace(0.0, wfs.t_span[1], 900)
live = {}
for name in COLORS:                                   # max |detuning| over the *live* rungs
    reach = np.zeros_like(scan)
    for tone in wfs.channels[name].tones:
        f = np.abs(np.asarray(tone.freq(scan - 0.5 * tau)))
        reach = np.maximum(reach, np.where(np.asarray(tone.env.A(scan - 0.5 * tau)) > 0.0, f, 0.0))
    live[name] = reach

fig, (ax_z, ax_p, ax_b) = plt.subplots(1, 3, figsize=(13.2, 3.8))
ax_z.plot(probes / us, requested_z / um, color="k", lw=3, alpha=0.25,
          label=r"requested $Z(t-\tau/2)$")
ax_z.plot(probes / us, zbar / um, color="#3a7bd5", lw=1.3, label=r"measured $\bar Z$")
ax_z.set(xlabel="t [µs]", ylabel="Z lab [µm]", title=r"$\bar Z$: 10 µm, held for a millisecond")
ax_z.legend(fontsize=8, loc="lower center")

ax_p.plot(probes / us, power / power.mean(), color="#8ac926", lw=1.2)
for centre in fade_times(wfs, "Ax", wfs.channels["Ax"].tones[0].env.g_centre):
    ax_p.axvline(centre / us, color="#3a7bd5", lw=0.7, alpha=0.35)
ax_p.axhline(1.0, color="k", lw=0.7)
ax_p.set(xlabel="t [µs]", ylabel="total tweezer power / mean",
         title=f"through {cycles:.0f} x hand-overs (blue): {100 * flatness:.2f} %")

for name, color in COLORS.items():                    # A wide, B thin: the pairs coincide
    ax_b.plot(scan / us, live[name] / MHz, color=color, lw=4.5 if name.startswith("A") else 1.6,
              label=name)
ax_b.plot(scan / us, np.abs(np.asarray(plain.channels["Ax"].tones[0].freq(scan - 0.5 * tau))) / MHz,
          color="#8d99ae", lw=1.3, ls=":", label="plain S19")
ax_b.axhline(10.0, color="k", lw=0.9, ls="--")
ax_b.annotate("→ 51 MHz", (470, 12.9), fontsize=8, color="#8d99ae")
ax_b.set(xlabel="t [µs]", ylabel="max |detuning| of a live rung [MHz]", ylim=(0, 15),
         title="band occupancy: bounded, not growing")
ax_b.legend(fontsize=8, ncols=2, loc="upper left")
plt.tight_layout()
plt.show()

## 4. What it costs: shadow tweezers

Mid-fade both $x$ channels carry two rungs, so the pupil product has four terms, not one
(Eq. S7 — the 16-ray picture of Fig. S6 with one axis fading). Two of them are co-located and
make the tweezer; the other two sit one rung off in the index *difference*, i.e. at
$\pm\,\text{deflection\_scale}\,\Delta f$. A shadow takes one factor from each pair
($\cos^{p_A}\theta\,\sin^{p_B}\theta$), so at the fade centre $\theta = \pi/4$ all four
combinations carry the same $2^{-(p_A+p_B)} = \tfrac12$ and **each shadow peaks at half the
trap** — not at the quarter a product of the two co-located intensities,
$(\cos\theta\sin\theta)^2$, would suggest.

The two shadows are *exactly* frequency-degenerate with each other, so `measure` reports them
as one group whose power-weighted centre lands at $x = 0$ — on top of the real tweezer, where
not one photon of *that* group is. Group position is meaningless for a degenerate pair; group
power is not, and their `power_coherent` equals their incoherent `power`, because 165 µm apart
their pupils do not overlap (§5 is the case where that fails).

For an array the same hand-over grows the grid to $(M_x{+}2)\times M_y$, and here Table II's
$p_B = 0$ matters: the $B$ ladder is a *rectangle*, so a new column does not fade up — it
**switches on at full brightness**. Two consequences for scheduling, both from the Supplement:

* a pick-up or hand-off must be timed in a non-fading zone, or the atom sees a companion trap
  appear beside it at half depth (single tweezer) or a whole new column at full depth (array);
* interlacing is exact only when the two axes share a $\Delta f$. An array needs
  $\Delta f_x \ne \Delta f_y$ so that its rows and columns stay distinguishable
  (`docs/conventions.md` §4), and the two fade schedules then beat against each other: some
  hand-overs happen on both axes at once and the full 16-ray grid lights up. §5 shows why that
  is survivable — unequal spacings also break the *frequency* degeneracy that makes the
  simultaneous scheme phase-sensitive.

How many extra columns appear is a question of **parity**, not of the scheme. A ladder of
spacing $\Delta f$ inside the $B$ window $(M+\eta)\Delta f$ holds $M$ or $M{+}1$ rungs
depending on where $f_Z$ sits, and the $A$ pair holds 1 or 2. For odd $M$ the two counts rise
together and the grid alternates $M \leftrightarrow M{+}2$ — the case above, and the one the
plan quotes. For **even** $M$ they alternate, so the grid is $M{+}1$ wide at *every* instant:
either $M{+}1$ equal columns (the $A$ pair on its plateau) or $M{-}1$ full ones plus two edges
trading. §6 measures exactly that on the 10×10.

In [ ]:
equal = ShepardConfig(DFX, DFX)                   # equal spacings: interlacing tiles exactly
even = synthesize(hold, P1, shepard=equal)
offset = P.deflection_scale * DFX
centre_t = float(fade_times(even, "Ax", even.channels["Ax"].tones[0].env.g_centre)[1])
plateau_t = float(fade_times(even, "Ax", 0.0)[1])

strip = FrameGrid(-1.6 * offset, 1.6 * offset, 1401, -2.5 * optics.waist0, 2.5 * optics.waist0, 21)
frames = {}
for label, probe in (("mid-fade", centre_t), ("plateau", plateau_t)):
    run = simulate(even, [probe])
    frames[label] = run.frame(0, strip, z_lab=float(np.asarray(z_hold(probe - 0.5 * tau))))
peak = max(float(f.max()) for f in frames.values())

terms = build_terms(even, centre_t)
trap = sum(m.power for m in measure(at_column(terms, 0.0, tol), optics))
shadows = [sum(m.power for m in measure(at_column(terms, side * offset, tol), optics))
           for side in (-1.0, 1.0)]
pair_group = max(measure(terms, optics), key=lambda m: m.power)
print(f"mid-fade rays        {terms.n_terms} (2 x 2 on x; the y pair is on its plateau)")
print(f"shadow positions     +-{offset / um:.1f} um = deflection_scale x delta_f_x")
print(f"shadow / trap power  {shadows[0] / trap:.4f}, {shadows[1] / trap:.4f}  (1/2 each, up to "
      f"the section 7a correction)")
print(f"the degenerate pair reports x = {pair_group.x / um:+.1e} um - where nothing is - and "
      f"coherent/incoherent = {pair_group.power_coherent / pair_group.power:.6f}")
assert all(abs(s / trap - 0.5) < 0.005 for s in shadows)

In [ ]:
DF_A = 1.0 * MHz
array_hold = TrajectorySpec(
    array=ArraySpec(3, 3, DF_A, DF_A),
    moves=(Lift(6 * um, 60 * us), Hold(150 * us), Lift(-6 * um, 60 * us)),
)
array_wfs = synthesize(array_hold, P1, shepard=ShepardConfig(DF_A, DF_A))
pitch = P.deflection_scale * DF_A


def middle_row(terms):
    "Brightness of each column of the array's middle row, relative to the brightest trap."
    xc, yc, _, _ = spot_params(terms, optics, 0.0)
    nodes = {}
    for i, j, c in zip(np.round(xc / pitch), np.round(yc / pitch), terms.c):
        nodes[(int(i), int(j))] = nodes.get((int(i), int(j)), 0.0) + float(abs(c) ** 2)
    reference = max(nodes.values())
    return [nodes.get((i, 0), 0.0) / reference for i in (-2, -1, 0, 1, 2)]


array_t = np.linspace(tau, array_hold.duration, 240)
trace = np.array([middle_row(build_terms(array_wfs, float(probe))) for probe in array_t])
print(f"columns lit            {sorted({int((row > 0).sum()) for row in trace})} of Mx = 3")
print(f"interior columns       {trace[:, 1:4].min():.6f} .. {trace[:, 1:4].max():.6f} of full")
print(f"brightest extra column {trace[:, [0, 4]].max():.4f} of full - it switches on, it does "
      f"not fade in")
assert np.allclose(trace[:, 1:4], 1.0, rtol=1e-12)
assert trace[:, [0, 4]].max() > 0.99

fig, (ax_i, ax_c) = plt.subplots(2, 1, figsize=(11.0, 5.8),
                                 gridspec_kw={"height_ratios": [1.15, 1]})
ax_i.imshow(np.maximum(np.vstack([frames["plateau"], frames["mid-fade"]]) / peak, 1e-9),
            extent=[strip.x0 / um, strip.x1 / um, 0, 2], origin="lower", aspect="auto",
            cmap="inferno", norm=LogNorm(vmin=1e-4, vmax=1.0))
ax_i.axhline(1.0, color="w", lw=0.8)
for side in (-1, 1):
    ax_i.annotate(f"{side * offset / um:+.0f} µm", (side * offset / um, 1.55), color="w",
                  ha="center", fontsize=8)
ax_i.set(yticks=[0.5, 1.5], yticklabels=["plateau", "mid-fade"], xlabel="X [µm]",
         title="one tweezer, log intensity (X and Y to different scales): the Eq. S31 "
               "companions live only inside a fade zone")

for i, (index, color) in enumerate(zip((-2, -1, 0, 1, 2),
                                       ("#c1121f", "#8ac926", "#3a7bd5", "#8ac926", "#c1121f"))):
    ax_c.plot(array_t / us, trace[:, i] + 0.004 * (i - 2), color=color, lw=1.4,
              ls="--" if index in (-1, 1) else "-",
              label={-2: "extended columns ±2", -1: "interior ±1", 0: "centre"}.get(index))
ax_c.set(xlabel="t [µs]", ylabel="column brightness / full", ylim=(-0.05, 1.15),
         title=r"3×3 array ($M_x$ odd): two extra columns switch on at full brightness, "
               r"then trade")
ax_c.legend(fontsize=8, ncols=3, loc="center right")
plt.tight_layout()
plt.show()

## 5. Why the fading is interlaced

Fade both axes together and four of the sixteen rays land **on the tweezer**: $(a,a|a,a)$,
$(a{-}1,a{-}1|a,a)$, $(a,a|a{-}1,a{-}1)$ and $(a{-}1,a{-}1|a{-}1,a{-}1)$. Their optical
frequencies follow the *sum* of the rung indices, so the two mixed rays — one axis old, the
other new — are exactly degenerate: a two-arm interferometer sitting on the trap, with no
control over its relative phase. That is the static Mach–Zehnder of Fig. S6.

The knob below is a phase $\varphi^{(n)} = n\,\Delta\varphi$ on the `By` ladder: linear in rung
index is linear in frequency, i.e. a **delay** — exactly what an acoustic or optical path
mismatch imposes (Eq. S29's $x_{\rm err}$, which the rest of this package assumes away). The
incoherent `power` cannot see it at all; `SpotMetrics.power_coherent` (the exact Gram form,
`field/measure.py`) can, and it swings the trap between ½ and 1½ of its nominal depth.

Interlacing ($\xi_y = ½$) leaves the other axis on its plateau, so the tweezer is fed by two
rays two ladder steps apart in frequency: they add in intensity and the sweep does nothing.

In [ ]:
simultaneous = {"Ay": ChannelFade(1, 0.5, 0.0), "By": ChannelFade(1, 0.5, 0.0)}
offsets = np.linspace(0.0, 2 * np.pi, 25)
fringe = {}
for label, config in (("interlaced (Table II)", "auto"), ("simultaneous", simultaneous)):
    cfg = ShepardConfig(DFX, DFX, config=config)
    reference = synthesize(hold, P1, shepard=cfg)
    n_rungs = reference.channels["By"].n_tones
    probe = float(fade_times(reference, "Ax", reference.channels["Ax"].tones[0].env.g_centre)[1])
    incoherent, coherent = [], []
    for phi in offsets:
        drive = synthesize(hold, P1, shepard=cfg, phases={"By": phi * np.arange(n_rungs)})
        spots = measure(at_column(build_terms(drive, probe), 0.0, tol), optics)
        incoherent.append(sum(m.power for m in spots))
        coherent.append(sum(m.power_coherent for m in spots))
    fringe[label] = (np.array(incoherent), np.array(coherent))
    print(f"{label:22s} rays in the frame {build_terms(drive, probe).n_terms:2d}, "
          f"coherent power swings {100 * np.ptp(coherent) / np.mean(incoherent):6.2f} % "
          f"({min(coherent) / np.mean(incoherent):.2f} .. "
          f"{max(coherent) / np.mean(incoherent):.2f} of the incoherent reading)")
assert np.ptp(fringe["simultaneous"][1]) / fringe["simultaneous"][0].mean() > 0.1
assert np.ptp(fringe["interlaced (Table II)"][1]) / fringe["interlaced (Table II)"][0].mean() < 1e-3

fig, ax = plt.subplots(figsize=(6.6, 3.9))
for (label, (inc, coh)), color in zip(fringe.items(), ("#3a7bd5", "#c1121f")):
    ax.plot(offsets / np.pi, coh / inc.mean(), color=color, lw=1.8, label=f"{label}: coherent")
    ax.plot(offsets / np.pi, inc / inc.mean(), color=color, lw=1.0, ls=":",
            label=f"{label}: incoherent")
ax.set(xlabel=r"per-rung phase offset on $B_y$  [$\pi$ rad]",
       ylabel="trap power / incoherent mean", ylim=(-0.05, 1.85),
       title="a path offset the experiment does not control (Fig. S6)")
ax.legend(fontsize=8, loc="lower center", ncols=2)
plt.tight_layout()
plt.show()

## 6. The user story, unhurried

Notebook 04 ran the product brief — *"move this 10×10 array from A to B, lifting 10 µm out of
plane on the way"* — in 80 µs, because that is all Eq. 1 left once the two ladders and the
lateral term had taken their share of the band. The comfortable version, 150 / 250 / 150 µs,
was refused outright. Here it is, synthesized by `shepard="auto"`: the band check runs first,
fails, and the synthesizer switches to fading-Shepard ladders and says so in its description.

Nothing about the array changes — for a ladder axis the Shepard ladder *is* the array ladder
(Eq. S27), so the same $\Delta f$ describes both. What changes is that all four channels now
carry a ladder instead of a tone, and that every rung is switched off before it can leave the
band.

The tracking check is per **term**, not per group: with a fading $A$ ladder the rung pairs
$(a, b)$ and $(a{-}1, b{+}1)$ share an optical frequency while sitting two columns apart, so a
group's centroid is not a trap position (§4). What must hold is that every spot sits on the
requested lattice — and it does, to $10^{-13}$ waists.

Two things about this grid an experiment has to plan for, both of them consequences of
$M = 10$ being **even** (§4):

* the array is $11\times11$ at every instant, not $10\times10$: the two edge columns trade
  brightness from one side to the other — together they carry one interior column's worth while
  the $A$ pair hands over, and two while it sits on its plateau — with the interior flat
  throughout;
* a Shepard ladder's rungs sit at *integer* multiples of $\Delta f$ from the lateral term
  (the trap positions depend on the index difference, so the $\xi$ offsets cancel), whereas an
  even-$M$ Eq. S19 array sits at half-integer multiples. Synthesis closes that half-pitch gap
  by adding a constant $\Delta f/2$ to the even-$M$ $B$ rung **frequencies** — and to nothing
  else, so $g$, the windows and the hand-over schedule are untouched
  (`waveform/shepard.py`, `lattice_comb_offset`). Both modes therefore put the traps in the
  same places, which is why the lattice below is anchored half a pitch off the array centre:
  for even $M$ that is where Eq. S19 puts the columns. Odd $M$ never needed the correction.

In [ ]:
story = TrajectorySpec(
    array=ArraySpec(10, 10, delta_f_x=1.0 * MHz, delta_f_y=1.3 * MHz),
    moves=(Lift(10 * um, 150 * us), Translate(40 * um, 25 * um, 250 * us), Lift(-10 * um, 150 * us)),
)
story_wfs = synthesize(story, P1, shepard="auto")
print("rungs: " + ", ".join(f"{n}:{cw.n_tones}" for n, cw in story_wfs.channels.items())
      + f"   ({story_wfs.n_tones} tones)")
head, _, reason = story_wfs.description.partition("[shepard='auto': ")
print(head.strip())
print("auto: " + reason.split(":")[0] + " (Eq. 1 again - now it is a ladder's problem, not a tone's)")

x_req, y_req, z_req = story.compile()
pitch_x, pitch_y = story.array.pitch(P)
# Even M: Eq. S19 puts the columns at half-integer multiples of the pitch, and since M5 the
# Shepard comb carries the delta_f/2 offset that lands them there too. So the lattice sits half
# a pitch off the array centre - anchor the node indices on the lattice, not on the centre.
half_x = 0.5 * pitch_x if story.array.mx % 2 == 0 else 0.0
half_y = 0.5 * pitch_y if story.array.my % 2 == 0 else 0.0
story_t = np.linspace(tau, story.duration + tau, 60)
residual, zerr, zbar_story, spread, astig_story, nodes_lit = [], [], [], [], [], []
edges = []
for probe in story_t:
    t_ret = float(probe) - 0.5 * tau
    terms = build_terms(story_wfs, float(probe))
    xc, yc, _, _ = spot_params(terms, optics, 0.0)
    dx, dy = xc - float(x_req(t_ret)) - half_x, yc - float(y_req(t_ret)) - half_y
    ix, iy = np.round(dx / pitch_x), np.round(dy / pitch_y)
    residual.append(max(np.max(np.abs(dx - ix * pitch_x)), np.max(np.abs(dy - iy * pitch_y))))
    nodes_lit.append((len(np.unique(ix)), len(np.unique(iy))))
    weight = np.abs(terms.c) ** 2
    column = np.array([weight[(ix == i) & (iy == 0)].sum() for i in (-5, 0, 5)])
    edges.append(column[[0, 2]] / column[1])
    z_axis = Z_LAB_SIGN * 2 * optics.focal_length ** 2 * terms.theta2 / optics.k
    z_lab = 0.5 * (z_axis[0] + z_axis[1])
    zbar_story.append(float(np.mean(z_lab)))
    zerr.append(float(np.max(np.abs(z_lab - float(z_req(t_ret))))))
    spread.append(float(z_lab.max() - z_lab.min()))
    astig_story.append(float(np.max(np.abs(z_axis[0] - z_axis[1]))))
residual, zerr, zbar_story, spread, astig_story, edges = (
    np.array(v) for v in (residual, zerr, zbar_story, spread, astig_story, edges))

print(f"\ngrid lit                {min(nodes_lit)[0]} x {min(nodes_lit)[1]} nodes at every probe "
      f"(Mx = My = 10, even: see section 4)")
print(f"edge columns            {edges.min():.2f} .. {edges.max():.2f} of an interior one; the "
      f"two of them sum to {edges.sum(axis=1).min():.2f} while the A pair hands over and to "
      f"{edges.sum(axis=1).max():.2f} while it sits on its plateau")
print(f"lattice residual        {residual.max() / optics.waist0:.2e} waists")
print(f"axial error             {zerr.max() / optics.rayleigh:.2e} z_R")
print(f"per-trap Z spread       {spread.max() / optics.rayleigh:.2e} z_R")
print(f"astigmatism             {astig_story.max() / optics.rayleigh:.2e} z_R")
assert residual.max() < 0.01 * optics.waist0
assert zerr.max() < 0.02 * optics.rayleigh
assert astig_story.max() < 0.02 * optics.rayleigh
assert set(nodes_lit) == {(11, 11)}

In [ ]:
fig, (ax_xy, ax_z, ax_e) = plt.subplots(1, 3, figsize=(13.2, 3.8))
t_ret = story_t - 0.5 * tau
for probe, color, label in ((story_t[0], "#3a7bd5", "t = tau"), (story_t[-1], "#c1121f", "at rest")):
    xc, yc, _, _ = spot_params(build_terms(story_wfs, float(probe)), optics, 0.0)
    ret = float(probe) - 0.5 * tau
    edge = ((np.abs(np.round((xc - float(x_req(ret)) - half_x) / pitch_x)) == 5)
            | (np.abs(np.round((yc - float(y_req(ret)) - half_y) / pitch_y)) == 5))
    ax_xy.scatter(xc[~edge] / um, yc[~edge] / um, s=6, color=color, label=label)
    ax_xy.scatter(xc[edge] / um, yc[edge] / um, s=14, facecolors="none", edgecolors=color, lw=0.6,
                  label=f"{label}: extended nodes")
ax_xy.plot(np.asarray(x_req(t_ret)) / um, np.asarray(y_req(t_ret)) / um, color="k", lw=1.2,
           alpha=0.5, label="requested centre")
ax_xy.set(xlabel="X [µm]", ylabel="Y [µm]", title="the grid, translated rigidly (11 × 11 nodes)")
ax_xy.set_aspect("equal")
ax_xy.legend(fontsize=7, loc="lower right")

ax_z.plot(story_t / us, np.asarray(z_req(t_ret)) / um, color="k", lw=3, alpha=0.25,
          label=r"requested $Z(t-\tau/2)$")
ax_z.plot(story_t / us, zbar_story / um, color="#3a7bd5", lw=1.3, label=r"measured $\bar Z$")
ax_z.set(xlabel="t [µs]", ylabel="Z lab [µm]", title="150 / 250 / 150 µs, at 10 µm")
ax_z.legend(fontsize=8, loc="lower center")

ax_e.semilogy(story_t / us, np.maximum(residual / optics.waist0, 1e-18), color="#8ac926", lw=1.3,
              label="lattice residual [waists]")
ax_e.semilogy(story_t / us, np.maximum(zerr / optics.rayleigh, 1e-18), color="#3a7bd5", lw=1.3,
              label=r"$\bar Z$ error [$z_R$]")
ax_e.semilogy(story_t / us, np.maximum(astig_story / optics.rayleigh, 1e-18), color="#c1121f",
              lw=1.3, label=r"$|\Delta F|$ [$z_R$]")
ax_e.set(xlabel="t [µs]", ylabel="error", ylim=(1e-17, 1e-2),
         title="the geometry is exact to machine precision")
ax_e.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

### Movie 05

The closing argument: the move notebook 04 had to refuse, at the pace an experiment would
actually ask for. Tracked view (the XY plane follows the scene's own best focus), hue carrying
each group's lab $Z$, the XZ slice beside it and the four channel drives underneath — where the
ladders' hand-overs show as fading lines.

One honest caveat about what you see: a hand-over takes $\Delta f/\dot f_Z = 20.6$ µs on $x$
and 26.8 µs on $y$ while the array is up, but these frames are 9.7 µs apart, so the edge
columns flicker rather than pulse. §4 is where they are held still.

In [ ]:
from IPython.display import Video

movie_frames = np.linspace(0.0, story_wfs.t_span[1], 60)
movie_run = simulate(story_wfs, movie_frames)
grid = auto_grid(movie_run, long_side=260)
print(f"grid {grid.nx} x {grid.ny} px over {(grid.x1 - grid.x0) / um:.0f} x "
      f"{(grid.y1 - grid.y0) / um:.0f} um, {len(movie_run.metrics[20])} groups per frame")

start = time.perf_counter()
movie = render_movie(movie_run, OUT / "05_shepard_story.mp4", grid=grid, mode="tracked", fps=15,
                     xz_shape=(104, 72), spectrogram_panel=True, dpi=100)
print(f"{movie}  ({movie.stat().st_size / 1e3:.0f} kB, {time.perf_counter() - start:.0f} s)")
Video(str(movie), embed=True, html_attributes="controls loop")

## 7. Two design caveats, quantified

**(a) How fast may a fade be?** The device layer describes each line's envelope by the
degree-2 Taylor expansion of Eq. S5 across the aperture, $\alpha = (1, -sA'/v, A''/2v^2)$. That
is exact only while the envelope changes slowly compared with the beam transit, and the
dimensionless measure of "slowly" is

$$\rho = \frac{w_{in}/v}{T_{\rm fade}},\qquad T_{\rm fade} = \frac{\eta\,\Delta f}{|\dot f_Z|},$$

the fraction of a fade that happens while the sound crosses one input radius. The $\alpha_1$
tilt term then adds a power correction of order $\rho^2$ to a fading line — which is exactly
the ripple §3 measured. It is **physics, not error**: a fade fast enough to apodize the pupil
does change the trap's coupling. Sweeping $\Delta f$ sweeps $\rho$, and the ripple follows the
$\rho^2$ law over an order of magnitude; the 1 % crossing is what to design against, and it is
why `auto_config` gives the fade window as much of the band as it can spare — a wide ladder is
both cheaper (fewer rungs) and slower to fade.

**(b) The compression correction's envelope shape.** At `mixing_order=3` the crystal compresses
each fundamental by $\sim m^2/8$ (Eqs. S20–S22), and `device/mixing.py` folds that correction
into its own fundamental, so it inherits the fundamental's envelope shape $(l_1, l_2) = (A'/A,
A''/A)$ instead of the $A^3$ shape its amplitude implies (multiplicity 3, i.e. $3l_1$). The
resulting pupil error, weighted by the size of the correction itself and averaged over the
input Gaussian, is

$$\varepsilon \;=\; \frac{m^2}{8}\sqrt{x^2 + \tfrac{3}{16}\big(3x^2 + y\big)^2},\qquad
x = \frac{l_1 w_{in}}{v},\quad y = \frac{l_2 w_{in}^2}{v^2},$$

which is $\approx (m^2/8)\,x$ wherever the tilt term dominates. WO-09 measured $\sim
1.2\times10^{-3}$ in the M2 mid band, i.e. $x \approx 0.1$ — and a fade shoulder is steepest
right at its outer edge, where the irising clamp of `FadeZoneEnvelope` pins
$x_{\rm max} = p\,\text{SLOPE\_CLAMP}\,(\pi/2)\rho$. For the 8 MHz ladder of §3 that is 0.09:
the same regime M2 already lives in, and inside the $x \lesssim 0.1$ bound where the
approximation is exact to better than a part in $10^3$. Narrow the ladder and it degrades on
the same $\rho$ axis as (a) — one parameter governs both.

In [ ]:
fdot_z = 10 * um / (2 * P.lens_scale)                 # the co-chirp a 10 um hold costs (Eq. 1)
sweep = []
for df in (2.0, 4.0, 6.0, 8.0, 12.0):
    cfg = ShepardConfig(df * MHz, df * (DFY / DFX) * MHz)
    drive = synthesize(hold, P1, shepard=cfg)
    when = np.unique(np.concatenate([
        fade_times(drive, ch, level) for ch in ("Ax", "Ay")
        for level in (0.0, drive.channels[ch].tones[0].env.g_centre)]))
    when = when[when <= hold.duration]
    tol_df = 0.25 * P.deflection_scale * cfg.delta_f_y
    p = np.array([sum(m.power for m in measure(at_column(build_terms(drive, float(s)), 0.0, tol_df),
                                               optics)) for s in when])
    rho = (optics.w_in / P.sound_speed) * fdot_z / (0.5 * df * MHz)
    sweep.append((rho, (p.max() - p.min()) / p.mean(), df))
sweep = np.array(sweep)

slope, intercept = np.polyfit(np.log(sweep[:, 0]), np.log(sweep[:, 1]), 1)
crossing = float(np.exp((np.log(0.01) - intercept) / slope))
df_crossing = (optics.w_in / P.sound_speed) * fdot_z / (0.5 * crossing)
print(f"flatness ~ rho^{slope:.2f}   ->   1 % at rho = {crossing:.3f}, i.e. delta_f = "
      f"{df_crossing / MHz:.1f} MHz")
for rho, flat, df in sweep:
    print(f"   delta_f = {df:4.1f} MHz   rho = {rho:.4f}   flatness = {100 * flat:6.3f} %")
assert 1.8 < slope < 2.2

In [ ]:
def pupil_error(l1, l2, m=P.channels["Ax"].drive_strength):
    "RMS relative pupil error of the compression correction's envelope shape (caveat b)."
    x = np.asarray(l1) * optics.w_in / P.sound_speed
    y = np.asarray(l2) * (optics.w_in / P.sound_speed) ** 2
    return (m ** 2 / 8) * np.sqrt(x ** 2 + (3 / 16) * (3 * x ** 2 + y) ** 2)


shoulders = {}
for label, df in ((f"{DFX / MHz:.0f} MHz ladder (section 3)", DFX), ("2 MHz ladder", 2.0 * MHz)):
    drive = synthesize(hold, P1, shepard=ShepardConfig(df, df))
    env = drive.channels["Ax"].tones[len(drive.channels["Ax"].tones) // 2].env
    when = np.linspace(float(env.crossing_times(env.g_inner)[0]),
                       float(env.crossing_times(env.g_outer)[0]), 400)
    amp = np.asarray(env.A(when))
    live_shoulder = amp > 0.0
    l1 = np.asarray(env.dA(when))[live_shoulder] / amp[live_shoulder]
    l2 = np.asarray(env.d2A(when))[live_shoulder] / amp[live_shoulder]
    shoulders[label] = (np.abs(l1) * optics.w_in / P.sound_speed, pupil_error(l1, l2))
    print(f"{label:28s} fade edge at l1 w_in / v = {shoulders[label][0].max():.3f}  ->  "
          f"pupil error {shoulders[label][1].max():.2e}")
print(f"{'M2 mid band (WO-09 finding 3)':28s} fade edge at l1 w_in / v = 0.100  ->  "
      f"pupil error {float(pupil_error(0.1 * P.sound_speed / optics.w_in, 0.0)):.2e}")

fig, (ax_r, ax_e) = plt.subplots(1, 2, figsize=(11.8, 3.9))
ax_r.loglog(sweep[:, 0], 100 * sweep[:, 1], "o-", color="#3a7bd5")
ax_r.axhline(1.0, color="k", lw=0.8, ls="--")
ax_r.axvline(crossing, color="#c1121f", lw=1.0)
ax_r.annotate(f"1 % at ρ = {crossing:.3f}\n({df_crossing / MHz:.1f} MHz)", (crossing, 3.0),
              fontsize=8, color="#c1121f", ha="left")
for rho, flat, df in sweep:
    ax_r.annotate(f"{df:.0f}", (rho, 100 * flat), textcoords="offset points", xytext=(4, -10),
                  fontsize=7, color="#3a7bd5")
ax_r.set(xlabel=r"$\rho = (w_{in}/v)\,/\,T_{fade}$", ylabel="power flatness [%]",
         title=r"(a) fast fades cost flatness as $\rho^2$   (labels: $\Delta f$ / MHz)")

for (label, (x_axis, err)), color in zip(shoulders.items(), ("#3a7bd5", "#f4a261")):
    order = np.argsort(x_axis)
    ax_e.loglog(x_axis[order], err[order], color=color, lw=1.7, label=label)
grid_x = np.logspace(-3, 0, 200)
ax_e.loglog(grid_x, (P.channels["Ax"].drive_strength ** 2 / 8) * grid_x, color="#8d99ae", lw=1.0,
            ls="--", label=r"tilt-only asymptote $m^2x/8$")
ax_e.axhline(1.2e-3, color="k", lw=0.8, ls=":")
ax_e.axvline(0.1, color="#c1121f", lw=1.0)
ax_e.annotate("WO-09 mid band", (2.2e-3, 1.5e-3), fontsize=8)
ax_e.set(xlabel=r"$l_1 w_{in} / v$", ylabel="relative pupil error", ylim=(1e-8, 1e-2),
         title="(b) compression-correction envelope approximation")
ax_e.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

## Where this leaves the package

Milestone 4 closes the last physical limit in the brief. The waveform layer can now express
an arbitrary 3D trajectory — including an axial offset held for as long as the experiment
wants — inside a fixed RF band, and the simulator resolves what that costs: shadow tweezers
at $\pm(\lambda F/v)\Delta f$ while a pair hands over, an array grid two columns wider, and a
sub-percent power ripple set by how fast the fade is compared with the beam transit.

Two numbers to design with:

* **$\Delta f$ as wide as the band allows.** It buys fewer rungs *and* a slower fade; at the
  default hardware the 1 % flatness crossing sits near 5 MHz for a 10 µm hold.
* **Never schedule a pick-up inside a fade zone.** Half-depth companions (single tweezer) or a
  whole extra column at full depth (array) appear there, and the array's extended columns
  switch on discontinuously.

What M5 adds is product surface, not physics: the `api.py` front door, a trajectory DSL, AWG
rendering of the parametric waveform files, and a report naming the band usage and the
predicted shadow schedule for a given move — the fade timetable this notebook computes by
hand in §4.